# 05 — Загрузка gold → ClickHouse (нативно, без Spark)**Архитектурный канон.** 
Spark своё дело сделал — построил витрины gold в озере.Дальше Spark **не нужен**: ClickHouse читает Parquet из MinIO **сам**, черезтабличную функцию `s3()`. Гонять данные через Spark-JDBC — антипаттерн (лишнийхоп, медленно). ClickHouse спроектирован тянуть Parquet из S3 напрямую.Команды в ClickHouse шлём по HTTP прямо из ноутбука (`requests`) — никакихдрайверов и ручного терминала. В проде эту SQL-загрузку дёргал бы оркестратор(Airflow); здесь её роль выполняет ноутбук, но механизм тот же — программный вызов.Запускать после `04_gold`.

## 1. Подключение к ClickHouse по HTTP

In [1]:
import os
import requests

CH_HOST = os.environ.get("CLICKHOUSE_HOST", "clickhouse")
CH_PORT = os.environ.get("CLICKHOUSE_PORT", "8123")
CH_USER = os.environ.get("CLICKHOUSE_USER", "spark")
CH_PASS = os.environ.get("CLICKHOUSE_PASSWORD", "spark")
CH_URL = f"http://{CH_HOST}:{CH_PORT}/"

def ch(sql):
    """Выполнить SQL в ClickHouse по HTTP. При ошибке показывает ответ сервера."""
    r = requests.post(CH_URL, params={"user": CH_USER, "password": CH_PASS}, data=sql.encode("utf-8"))
    if r.status_code != 200:
        raise RuntimeError(f"ClickHouse {r.status_code}:\n{r.text}")
    return r.text

print(ch("SELECT version()").strip(), "- ClickHouse на связи")

26.5.1.882 - ClickHouse на связи


## 2. Создаём базу gold
ClickHouse читает Parquet из MinIO сам. Адрес MinIO внутри сети — `minio:9000`.

In [2]:
ch("CREATE DATABASE IF NOT EXISTS gold")
print("База gold готова")

База gold готова


## 3. Загружаем витрины из озера
Для каждой витрины: `CREATE TABLE ... AS SELECT * FROM s3(...)`.ClickHouse идёт в MinIO, читает Parquet по маске `*.parquet`, выводит схемуи создаёт таблицу MergeTree. Delta-данные лежат как `*.parquet` (служебный`_delta_log` содержит `.json`, под маску не попадает).

In [3]:
MINIO = "http://minio:9000"
S3KEY, S3SECRET = "minioadmin", "minioadmin"

# Spark пишет колонки как Nullable, а ClickHouse не пускает Nullable
# в ключ сортировки (ORDER BY). Поэтому ключевые колонки оборачиваем
# в assumeNotNull() — по смыслу витрины они всегда заполнены.
def load_mart(table, order_by, key_cols):
    src = f"s3('{MINIO}/gold/{table}/*.parquet', '{S3KEY}', '{S3SECRET}', 'Parquet')"
    # ключевые колонки -> NOT NULL, остальные как есть
    if key_cols:
        not_null = ", ".join(f"assumeNotNull({col}) AS {col}" for col in key_cols)
        others = f"* EXCEPT ({', '.join(key_cols)})"
        select = f"SELECT {not_null}, {others} FROM {src}"
    else:
        select = f"SELECT * FROM {src}"
    ch(f"DROP TABLE IF EXISTS gold.{table}")
    ch(f"CREATE TABLE gold.{table} ENGINE = MergeTree() ORDER BY {order_by} AS {select}")
    cnt = ch(f"SELECT count() FROM gold.{table}").strip()
    print(f"  {table}: {cnt} строк -> ClickHouse")

print("Загрузка витрин из озера в ClickHouse:")
load_mart("revenue_by_segment", "tuple()", [])
load_mart("top_clients", "(city, rank_in_city)", ["city", "rank_in_city"])
load_mart("daily_transactions", "dt", ["dt"])
print("Готово")

Загрузка витрин из озера в ClickHouse:
  revenue_by_segment: 3 строк -> ClickHouse
  top_clients: 45 строк -> ClickHouse
  daily_transactions: 2 строк -> ClickHouse
Готово


## 4. Проверка: читаем витрину из ClickHouse
Это уже чтение из ClickHouse — так же будет читать Metabase.`FORMAT` делает вывод аккуратной таблицей.

In [4]:
print(ch('''
    SELECT segment, total_revenue, tx_count, avg_amount
    FROM gold.revenue_by_segment
    ORDER BY total_revenue DESC
    FORMAT PrettyCompact
'''))

   ┌─segment──┬─total_revenue─┬─tx_count─┬─avg_amount─┐
1. │ mass     │ 1581878437.11 │    63222 │   25021.01 │
2. │ affluent │  538541326.82 │    21662 │   24861.11 │
3. │ private  │  521293501.62 │    21020 │   24799.88 │
   └──────────┴───────────────┴──────────┴────────────┘



## Готово
Витрины в ClickHouse — нативно из озера, без Spark-посредника. Это канон:ClickHouse сам читает Parquet из S3. Данные готовы к визуализации.Дальше — `06`: Metabase подключается к ClickHouse и строит дашборды по URL.